# Anomaly Detection — Extended: Autoencoder + Deep Evaluation
## Toy Neural Network · LOF · ROC/PR Curves · Feature Importance · Model Persistence

This notebook extends the anomaly detection pipeline from notebooks **0** and **1** with:

1. **Local Outlier Factor** — a third classical detector for comparison
2. **Toy Neural Network Autoencoder** — built from scratch in PyTorch; learns to reconstruct normal signals and uses high reconstruction error as an anomaly score
3. **Rich evaluation** — ROC curves, PR curves, normalized confusion matrices for all three models side-by-side
4. **Hyperparameter sensitivity** — how does LOF's `n_neighbors` affect performance?
5. **Permutation feature importance** — which of our 5 features matters most?
6. **Model persistence** — save trained models and results to disk

### What you'll learn
- How autoencoders work and why they're natural anomaly detectors
- How to implement a PyTorch autoencoder and train it on normal data only
- How to interpret reconstruction error as an anomaly score
- How to produce publication-quality evaluation plots

### Prerequisites
Notebooks **0** or **1** (for context). Basic Python and NumPy. PyTorch for the autoencoder section.

---
*Algorithms: OC-SVM · Isolation Forest · LOF · Autoencoder (PyTorch)*

In [ ]:
# ── Imports & Setup ──────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    classification_report, f1_score, precision_score, recall_score,
    accuracy_score, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve, ConfusionMatrixDisplay,
    RocCurveDisplay, PrecisionRecallDisplay,
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import warnings, os, hashlib
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)
SEED = 42

plt.rcParams.update({
    'figure.dpi': 130, 'axes.titlesize': 13,
    'axes.labelsize': 11, 'font.size': 10, 'legend.fontsize': 9,
})
sns.set_style("whitegrid")

print(f"PyTorch {torch.__version__}")
print("All imports successful.")

In [ ]:
# ── Reproduce the same dataset + features from notebook 1 ────
fs = 10000; T_window = 0.5; N_samples = int(fs * T_window)
n_windows = 500; fault_ratio = 0.10
shaft_freq = 30; bpfi = 5.2 * shaft_freq
t = np.arange(N_samples) / fs

def generate_normal_vibration(t, shaft_freq, fs):
    sig = (1.0*np.sin(2*np.pi*shaft_freq*t) + 0.5*np.sin(2*np.pi*2*shaft_freq*t+0.3) +
           0.3*np.sin(2*np.pi*3*shaft_freq*t+1.1) + 0.15*np.sin(2*np.pi*4*shaft_freq*t+0.7))
    return sig + 0.4*np.random.randn(len(t))

def generate_faulty_vibration(t, shaft_freq, bpfi, fs):
    normal = generate_normal_vibration(t, shaft_freq, fs)
    impulse_period = 1.0 / bpfi
    impulses = np.zeros_like(t)
    for ti in np.arange(0, t[-1], impulse_period):
        idx = np.argmin(np.abs(t - ti))
        if idx < len(t) - 10:
            burst_len = min(150, len(t) - idx)
            burst = np.exp(-np.arange(burst_len)/20.0) * np.random.uniform(2.5, 4.5)
            carrier = np.sin(2*np.pi*3500*np.arange(burst_len)/fs)
            impulses[idx:idx+burst_len] += burst * carrier
    am = 1.0 + 0.3*np.sin(2*np.pi*bpfi*t)
    return normal * am + impulses

def extract_features(sig, fs):
    N = len(sig)
    rms = np.sqrt(np.mean(sig**2)); mean_abs = np.mean(np.abs(sig)); peak = np.max(np.abs(sig))
    freqs = np.fft.rfftfreq(N, 1/fs); fft_mag = np.abs(np.fft.rfft(sig))
    return [stats.kurtosis(sig, fisher=False), peak/rms if rms>0 else 0,
            stats.skew(sig), rms/mean_abs if mean_abs>0 else 0,
            np.sum(freqs*fft_mag)/np.sum(fft_mag) if np.sum(fft_mag)>0 else 0]

n_faulty = int(n_windows*fault_ratio); n_normal = n_windows - n_faulty
signals, labels = [], []
for _ in range(n_normal): signals.append(generate_normal_vibration(t, shaft_freq, fs)); labels.append(0)
for _ in range(n_faulty): signals.append(generate_faulty_vibration(t, shaft_freq, bpfi, fs)); labels.append(1)
signals, labels = np.array(signals), np.array(labels)
idx = np.random.permutation(n_windows); signals, labels = signals[idx], labels[idx]

feature_names = ['Kurtosis', 'Crest Factor', 'Skewness', 'Shape Factor', 'Spectral Centroid']
df = pd.DataFrame([extract_features(signals[i], fs) for i in range(n_windows)], columns=feature_names)
df['Label'] = labels; df['Label_str'] = df['Label'].map({0:'Normal', 1:'Fault'})

X = df[feature_names].values; y_true = df['Label'].values
scaler = StandardScaler(); X_scaled = scaler.fit_transform(X)
pca = PCA(n_components=2); X_pca = pca.fit_transform(X_scaled)

# Re-train baseline models
ocsvm = OneClassSVM(kernel='rbf', gamma='scale', nu=0.10).fit(X_scaled)
y_pred_ocsvm = (ocsvm.predict(X_scaled) == -1).astype(int)
scores_ocsvm = ocsvm.decision_function(X_scaled)

iforest = IsolationForest(n_estimators=200, contamination=0.10, random_state=SEED).fit(X_scaled)
y_pred_iforest = (iforest.predict(X_scaled) == -1).astype(int)
scores_iforest = iforest.decision_function(X_scaled)

print(f"Dataset ready: {n_windows} windows | Normal: {(labels==0).sum()} | Fault: {(labels==1).sum()}")